# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjds22/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a decision-tree classifier as a candidate model because it can learn simple, non-linear relationships between search-performance features and a defined outcome. The model will be compared with the Week 4 rule-based baseline using the same observations, target definition, data split and evaluation metric. The target must be defined from an observable outcome rather than an internal FlyRank flag, because those flags are not available in the released data. Model performance will be treated as exploratory decision support, not proof that recommended changes cause better search performance.

In [2]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
print("HF token loaded:", bool(HF_TOKEN))

HF token loaded: True


In [3]:

import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

print(con.sql(f"""
    SELECT month, COUNT(*) AS rows
    FROM {fact_daily}
    WHERE gsc_data_available = TRUE
    GROUP BY month
    ORDER BY month
""").df().to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  month    rows
2025-01    1297
2025-02   75985
2025-03  167859
2025-04  285114
2025-05  349923
2025-06  329201
2025-07  469794
2025-08  704962
2025-09  845813
2025-10 1187654
2025-11 1681368
2025-12 2087675
2026-01 2397143
2026-02 2621783
2026-03 3611061
2026-04 3901060
2026-05 4373422
2026-06 3878937


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a time-aware split to reduce the risk of future information leaking into training. Monthly content features from earlier periods predict whether the same content has fewer than 100 impressions in the following month. The final test period uses May 2026 features to predict June 2026 outcomes. Both the decision-tree model and the Week 4 rule-based baseline will be evaluated on the same held-out observations. This measures prediction of next-month low impressions, not the causal effect of content changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The model was trained on 150,000 sampled observations from earlier months and evaluated on 30,000 sampled observations using May 2026 features to predict June 2026 low impressions. On this test sample, the decision tree achieved an F1 score of 0.847, compared with 0.775 for the rule-based baseline. Its precision was 0.906 versus 0.688 for the baseline, while its recall was lower at 0.795 versus 0.889. This suggests the model is more selective when identifying content likely to have low impressions, but misses more positive cases. These findings are exploratory and apply to this sample and target definition; they do not show that content changes cause performance improvements.

In [4]:

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score
)

# Aggregate daily data into monthly content-level observations.
monthly = con.sql(f"""
    SELECT
        month,
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM {fact_daily}
    WHERE month >= '2025-01'
      AND month <= '2026-06'
      AND gsc_data_available = TRUE
    GROUP BY month, client_hash_id, content_hash_id
""").df()

monthly["month_date"] = pd.to_datetime(
    monthly["month"].astype(str), format="%Y-%m", errors="coerce"
)
monthly = monthly.dropna(subset=["month_date"])

# Match each month to the same content in the following calendar month.
future = monthly[
    ["client_hash_id", "content_hash_id", "month_date", "impressions"]
].copy()
future["month_date"] = future["month_date"] - pd.DateOffset(months=1)
future = future.rename(columns={"impressions": "next_impressions"})

data = monthly.merge(
    future,
    on=["client_hash_id", "content_hash_id", "month_date"],
    how="inner"
)

# Target: fewer than 100 impressions next month.
data["target"] = (data["next_impressions"] < 100).astype(int)

# Training uses feature months through April; testing uses May.
train = data[data["month_date"] <= "2026-04-01"].copy()
test = data[data["month_date"] == "2026-05-01"].copy()

# Fixed-size samples keep the exercise manageable.
train = train.sample(
    n=min(150000, len(train)), random_state=42
)
test = test.sample(
    n=min(30000, len(test)), random_state=42
)

features = ["impressions", "clicks", "avg_position"]

train = train.replace([np.inf, -np.inf], np.nan).dropna(
    subset=features + ["target"]
)
test = test.replace([np.inf, -np.inf], np.nan).dropna(
    subset=features + ["target"]
)

X_train = train[features]
y_train = train["target"]
X_test = test[features]
y_test = test["target"]

model = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=100, random_state=42,
    class_weight="balanced"
)
model.fit(X_train, y_train)

model_pred = model.predict(X_test)

# Week 4-style rule, applied to the same test observations.
baseline_pred = (
    (X_test["impressions"] < 100) |
    (X_test["avg_position"] >= 20)
).astype(int)

results = pd.DataFrame([
    {
        "Method": "Week 4 rule baseline",
        "Accuracy": accuracy_score(y_test, baseline_pred),
        "Precision": precision_score(y_test, baseline_pred, zero_division=0),
        "Recall": recall_score(y_test, baseline_pred, zero_division=0),
        "F1": f1_score(y_test, baseline_pred, zero_division=0)
    },
    {
        "Method": "Decision tree",
        "Accuracy": accuracy_score(y_test, model_pred),
        "Precision": precision_score(y_test, model_pred, zero_division=0),
        "Recall": recall_score(y_test, model_pred, zero_division=0),
        "F1": f1_score(y_test, model_pred, zero_division=0)
    }
])

print("Training observations:", len(train))
print("Test observations:", len(test))
print("Test feature month: May 2026")
print("Target outcome month: June 2026")
display(results.round(3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training observations: 150000
Test observations: 30000
Test feature month: May 2026
Target outcome month: June 2026


,Method,Accuracy,Precision,Recall,F1
0,Week 4 rule baseline,0.751,0.688,0.889,0.775
1,Decision tree,0.861,0.906,0.795,0.847


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The decision tree achieved higher precision and F1 than the rule-based baseline, but lower recall on the held-out sample. False positives are observations predicted to have low impressions that did not meet the target, while false negatives are low-impression observations the model missed. Feature importance indicates which input variables the fitted tree relied on, but does not establish causation. The model should therefore support prioritisation rather than automatically determine which content needs changing.

In [5]:

from sklearn.metrics import confusion_matrix

# Identify correct and incorrect predictions
errors = test[["impressions", "clicks", "avg_position", "target"]].copy()
errors["actual"] = y_test.to_numpy()
errors["predicted"] = model_pred
errors["error_type"] = np.select(
    [
        (errors["actual"] == 1) & (errors["predicted"] == 0),
        (errors["actual"] == 0) & (errors["predicted"] == 1)
    ],
    ["False negative", "False positive"],
    default="Correct"
)

print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, model_pred))

print("\nPrediction error counts:")
print(errors["error_type"].value_counts())

print("\nFeature importance:")
display(pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False).round(3))

print("\nExample false positives:")
display(errors[errors["error_type"] == "False positive"].head(5))

print("\nExample false negatives:")
display(errors[errors["error_type"] == "False negative"].head(5))


Confusion matrix [ [TN, FP], [FN, TP] ]:
[[14279  1195]
 [ 2984 11542]]

Prediction error counts:
error_type
Correct           25821
False negative     2984
False positive     1195
Name: count, dtype: int64

Feature importance:


,Feature,Importance
0,impressions,0.994
2,avg_position,0.006
1,clicks,0.000



Example false positives:


,impressions,clicks,avg_position,target,actual,predicted,error_type
1145591,101.0,0.0,20.546115,0,0,1,False positive
501133,77.0,0.0,10.069333,0,0,1,False positive
1103233,52.0,0.0,14.158333,0,0,1,False positive
516832,81.0,1.0,35.628333,0,0,1,False positive
1144085,71.0,1.0,17.706349,0,0,1,False positive



Example false negatives:


,impressions,clicks,avg_position,target,actual,predicted,error_type
1086910,235.0,0.0,46.864940,1,1,0,False negative
538057,151.0,0.0,40.888210,1,1,0,False negative
1097058,267.0,0.0,37.008771,1,1,0,False negative
500988,177.0,0.0,17.007101,1,1,0,False negative
489145,188.0,0.0,42.918179,1,1,0,False negative


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.